# 航空会社顧客満足度予測: 実験 #002 ドメイン特徴量エンジニアリング & LightGBM チューニング

- **コンペ名:** [Playground Series - Season 6, Episode 10](https://www.kaggle.com/competitions/playground-series-s6e10)
- **タスク:** 顧客満足度予測 (2値分類)
- **評価指標:** **ROC AUC**
- **前回のベースライン:** OOF ROC AUC: `0.95828` / LB: `0.95791` (1,697位)

---

## 今回の改善仮説 (Hypothesis)
1. **遅延の挽回 (Delay Diff):** 「出発時に遅延したがフライト中に挽回できたか、さらに悪化したか」は乗客の心理的ストレスに強く影響するはず。
2. **サービス最低点 (Min Score):** 「たった1つの致命的な不満サービス（1点）」があるだけで、乗客は全体を不満と評価しやすくなるはず。
3. **クラス × 目的の交差 (Class_Travel):** 「出張ビジネス客」と「個人エコノミー客」では満足の基準が根本的に異なるはず。
4. **パラメータチューニング:** 学習率を `0.03` に落とし、決定木の表現力（`num_leaves: 63`）を高めることで、より高精度な決定境界を引けるはず。

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

candidates = [
    '/kaggle/input/competitions/playground-series-s6e10',
    '/kaggle/input/playground-series-s6e10',
    '002_airline-satisfaction/data',
    '../data',
    'data'
]
data_dir = next((c for c in candidates if os.path.exists(os.path.join(c, 'train.csv'))), None)
if not data_dir:
    matches = glob.glob('**/playground-series-s6e10/train.csv', recursive=True)
    data_dir = os.path.dirname(matches[0]) if matches else '.'

print(f'データ取得元: {data_dir}')
train_df = pd.read_csv(os.path.join(data_dir, 'train.csv'))
test_df = pd.read_csv(os.path.join(data_dir, 'test.csv'))
sub_df = pd.read_csv(os.path.join(data_dir, 'sample_submission.csv'))

print(f'訓練データ: {train_df.shape}')
print(f'テストデータ: {test_df.shape}')

## 特徴量エンジニアリングの実行

In [ ]:
service_cols = [
    'Inflight wifi service', 'Departure/Arrival time convenient',
    'Ease of Online booking', 'Gate location', 'Food and drink',
    'Online boarding', 'Seat comfort', 'Inflight entertainment',
    'On-board service', 'Leg room service', 'Baggage handling',
    'Checkin service', 'Cleanliness'
]

def engineer_features(df):
    X = df.copy()
    dep_delay = X['Departure Delay in Minutes'].fillna(0)
    arr_delay = X['Arrival Delay in Minutes'].fillna(0)
    X['Total_Delay'] = dep_delay + arr_delay
    X['Has_Delay'] = (X['Total_Delay'] > 0).astype(int)
    X['Delay_Diff'] = arr_delay - dep_delay
    
    X['Service_Mean'] = X[service_cols].mean(axis=1)
    X['Service_Min']  = X[service_cols].min(axis=1)
    X['Service_Max']  = X[service_cols].max(axis=1)
    X['Service_Std']  = X[service_cols].std(axis=1)
    X['Service_Sum']  = X[service_cols].sum(axis=1)
    
    X['Comfort_Score'] = X['Seat comfort'] + X['Leg room service'] + X['Cleanliness']
    X['Digital_Score'] = X['Inflight wifi service'] + X['Ease of Online booking'] + X['Online boarding']
    X['Class_Travel'] = X['Class'].astype(str) + '_' + X['Type of Travel'].astype(str)
    
    drop_cols = [c for c in ['id', 'satisfaction'] if c in X.columns]
    X = X.drop(columns=drop_cols)
    for c in X.select_dtypes(include=['object', 'string']).columns:
        X[c] = X[c].astype('category')
    return X

X = engineer_features(train_df)
X_test = engineer_features(test_df)
y = train_df['satisfaction'].astype(int).values
print(f'特徴量拡張完了: {X.shape[1]} 列')
X.head(3)

## 5分割交差検証（Stratified 5-Fold CV）による学習

In [ ]:
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

oof_preds = np.zeros(len(train_df))
test_preds = np.zeros(len(test_df))
models = []

lgb_params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'learning_rate': 0.03,
    'num_leaves': 63,
    'max_depth': -1,
    'n_estimators': 1000,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1,
    'subsample': 0.9,
    'colsample_bytree': 0.9,
}

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_tr, y_tr = X.iloc[train_idx], y[train_idx]
    X_va, y_val = X.iloc[val_idx], y[val_idx]
    
    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
    )
    val_pred = model.predict_proba(X_va)[:, 1]
    oof_preds[val_idx] = val_pred
    score = roc_auc_score(y_val, val_pred)
    models.append(model)
    print(f'Fold {fold}/{n_splits} ROC AUC: {score:.5f}')
    
    test_preds += model.predict_proba(X_test)[:, 1] / n_splits

new_cv = roc_auc_score(y, oof_preds)
print('=' * 45)
print(f'新 OOF ROC AUC (手元検証): {new_cv:.5f} (前回ベースライン: 0.95828)')
print(f'改善スコア差分: {new_cv - 0.95828:+.5f}')
print('=' * 45)

## 特徴量重要度（Feature Importance）の確認
新しく追加したドメイン特徴量（Min Scoreや遅延差分など）がどれくらい効いているか可視化します。

In [ ]:
importances = np.mean([m.feature_importances_ for m in models], axis=0)
fi_df = pd.DataFrame({'feature': X.columns, 'importance': importances})
fi_df = fi_df.sort_values('importance', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(data=fi_df.head(20), x='importance', y='feature', palette='viridis')
plt.title('Top 20 Features Importance (Experiment #002)')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

## 提出ファイル作成

In [ ]:
sub_df['satisfaction'] = test_preds
sub_df.to_csv('submission_fe.csv', index=False)
print('提出ファイル作成完了: submission_fe.csv')
print(sub_df.head())